---
# unimodal ablation  Framework


> **Before Running**
>
> - Select the experiment configuration by setting **only one** of the following:
>   ```python
>   config = CONFIG_CLASS6   # 6-class classification
>   # config = CONFIG_CLASS8   # 8-class classification
>   ```
> - Update all dataset paths in `BASE_CONFIG`.
> - Verify the output directory before starting training.



### imports 

In [ ]:
# =========================================================
# IMPORTS
# =========================================================

# ---------------------------------------------------------
# Standard Library
# ---------------------------------------------------------
import os
import random
import shutil
import time
from collections import Counter
from datetime import datetime

# ---------------------------------------------------------
# Scientific Computing
# ---------------------------------------------------------
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
import yaml
from tabulate import tabulate
from copy import deepcopy
# ---------------------------------------------------------
# PyTorch
# ---------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
    random_split,
)

# ---------------------------------------------------------
# TorchVision
# ---------------------------------------------------------
from torchvision import transforms

from torchvision.models import (
    resnet18,
    resnet50,
    ResNet18_Weights,
    ResNet50_Weights,
)

from torchvision.models.vision_transformer import (
    vit_b_16,
    ViT_B_16_Weights,
)

from torchvision.models.segmentation import (
    deeplabv3_resnet50,
)

# ---------------------------------------------------------
# timm
# ---------------------------------------------------------
import timm
from timm import create_model

from timm.models.mobilevit import mobilevit_s

from timm.models.swin_transformer import (
    swin_tiny_patch4_window7_224,
    swin_tiny_patch4_window7_224_Weights,
)

# ---------------------------------------------------------
# Scikit-Learn
# ---------------------------------------------------------
from sklearn.model_selection import StratifiedShuffleSplit

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ---------------------------------------------------------
# Profiling
# ---------------------------------------------------------
from ptflops import get_model_complexity_info

## Configuration
- Configuration files, reproducibility utilities, and experiment settings.

### Global Seed

- Ensures deterministic and reproducible experiments.

In [ ]:


def set_global_seed(seed=101):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### Configuration 




In [ ]:
# ==========================================================
# CONFIGURATION
# ==========================================================


# ==========================================================
# BASE CONFIGURATION
# ==========================================================

BASE_CONFIG = {

    # ==================================================
    # GLOBAL
    # ==================================================
    "seeds": [45, 22, 13, 101, 220],

    # ==================================================
    # PATHS
    # ==================================================
    "paths": {
        "base_dir": "path to base directory for saving models and logs and results",
    },

    # ==================================================
    # DATA
    # ==================================================
    "data": {
        "train": {
            "thermal": "path to thermal data",
            "pose": "path to pose data",
            "labels": "path to labels data",
            "pose_mask": "path to pose mask data",
        }
    },

    # ==================================================
    # TRAINING
    # ==================================================
    "training": {

        "batch_size": 32,
        "epochs": 200,
        "lr": 0.0005,
        "eval_path": "",

        "patience": 20,
        "min_delta": 0.001,
        "min_epochs": 50,

        "sampler": "class_aware",

        # Loss
        "loss_type": "cb_focal",
        "enable_loss_switch": True,
        "loss_switch_patience": 10,

        "class_weights": None,
        "samples_per_class": None,

        "label_smoothing": 0.1,
        "ldam_margin": 0.5,

        # Ablations
        "enable_loss_ablation": False,
        "enable_classifier_ablation": False,
        "enable_multiseed_ablation": True,

        "default_seed": 45,
    },

    # ==================================================
    # INPUT
    # ==================================================
    "input": {
        "height": 224,
        "width": 224,
    },

    # ==================================================
    # MODEL
    # ==================================================
    "model": {

        "classifier": {
            "num_classes": None      # ← Filled automatically
        },

        "input_dim_projection": 128,
        "output_dim_fusion": 128,
        "input_dim_classifier": 128,

        "num_heads": 4,
        "num_layers": 2,
    },

    # ==================================================
    # EXPERIMENT SPACE
    # ==================================================
    "experiments": {

        # Thermal Encoders
        "thermal_encoders": [
            "EfficientFormerV2-S",
        ],

        # Pose Encoders
        "pose_modules": [
            "mlp",
        ],
        # Classifiers
        "classifiers": [
            "Cosine",
        ],

        # Losses
        "losses": [
            "cb_focal",
            "focal",
            "label_smooth",
            "ldam",
            "cost_sensitive",
            "ce",
        ],
    },
}


# ==========================================================
# CONFIG BUILDER
# ==========================================================

def get_config(num_classes: int):

    cfg = deepcopy(BASE_CONFIG)

    cfg["model"]["classifier"]["num_classes"] = num_classes

    return cfg


# ==========================================================
# PREDEFINED CONFIGS
# ==========================================================

CONFIG_CLASS6 = get_config(6)

CONFIG_CLASS8 = get_config(8)

### configuration validation

In [ ]:
# =========================================================
# CONFIG VALIDATION
# =========================================================


def validate_config(cfg):
    # ---- Top-level sections ----
    assert "data" in cfg, "Missing 'data' in config"
    assert "training" in cfg, "Missing 'training' in config"
    assert "model" in cfg, "Missing 'model' in config"


    # ---- Dataset paths ----
    assert "train" in cfg["data"], "Missing data.train"
    assert "thermal" in cfg["data"]["train"], "Missing data.train.thermal"
    assert "pose" in cfg["data"]["train"], "Missing data.train.pose"
    assert "labels" in cfg["data"]["train"], "Missing data.train.labels"
    assert "pose_mask" in cfg["data"]["train"], "Missing data.train.pose_mask"


    # ---- Training hyperparameters ----
    assert "epochs" in cfg["training"], "Missing training.epochs"
    assert "batch_size" in cfg["training"], "Missing training.batch_size"
    assert "lr" in cfg["training"], "Missing training.lr"


    # ---- Model definition ----
    assert "classifier" in cfg["model"], "Missing model.classifier"
    assert "num_classes" in cfg["model"]["classifier"], "Missing model.classifier.num_classes"



---
## Utilities

Reusable helper functions shared across the framework.


### Configuration Loader

Loads experiment configuration files.


In [ ]:

def load_config(config_path):
    with open(config_path, 'r') as file:
        return yaml.safe_load(file)




### Model Checkpoint Saver

Saves trained model weights.

In [ ]:

def save_model(model, path):
    torch.save(model.state_dict(), path)




### Metrics Computation

Computes evaluation metrics for classification.

In [ ]:

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "recall": recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "report": classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    }

### Cost Matrix Generation

Generates a cost matrix from the confusion matrix for cost-sensitive learning."

In [ ]:


def generate_cost_matrix_from_confusion(confusion_matrix):
    # Convert to float32 for stability
    cm = confusion_matrix.astype(np.float32)
    num_classes = cm.shape[0]

    # Default: all 1s with 0 diagonal
    cost_matrix = np.ones((num_classes, num_classes))
    np.fill_diagonal(cost_matrix, 0)

    # Normalize by misclassification rates
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid division by zero
    misclass_rate = cm / row_sums

    # Penalty = 1 - misclass rate
    penalty = 1.0 - misclass_rate
    np.fill_diagonal(penalty, 0)

    # Scale penalty to avoid exploding gradients 
    max_val = penalty.max()
    if max_val > 0:
        penalty /= max_val

    # Convert to tensor on GPU if available
    return torch.tensor(penalty, dtype=torch.float32).to(
        torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )


### Logging Utilities

In [ ]:
# =========================================================
# RESULT TABLE HEADERS (UNI-MODAL)
# =========================================================


headers = ["Model", "F1", "Precision", "Recall", "Accuracy", "Status"]
# =========================================================
# BASIC RESULT LOGGING (UNI-MODAL)
# =========================================================


def log_results(row, summary_log_path):
    table = tabulate([row], headers=headers, tablefmt="grid")
    with open(summary_log_path, "a") as f:
        f.write(table + "\n\n")




def log_profile_results(row, profile_log_path):
    profile_headers = ["Model Combo", "Params (M)", "GFLOPs", "Size (MB)", "Avg Inference (ms)"]
    table = tabulate([row], headers=profile_headers, tablefmt="grid")
    with open(profile_log_path, "a") as f:
        f.write(table + "\n\n")


def load_completed(progress_file):
    """
    Load completed run IDs from progress file.
    """
    if not os.path.exists(progress_file):
        return set()


    with open(progress_file, "r") as f:
        return set(
            line.strip()
            for line in f
            if line.strip()
        )
    

---

## Data

### Dataset

Dataset implementation for multimodal thermal and pose inputs.

#### Unimodal Dataset

Loads thermal images, pose sequences, masks, and labels.

In [ ]:
class UnimodalDataset(Dataset):
    def __init__(self, data_config, thermal_transform=None, pose_augment_fn=None, modality='thermal', indices=None):
        self.thermal = torch.load(data_config['thermal'])
        self.pose = torch.load(data_config['pose'])
        self.pose_mask = torch.load(data_config['pose_mask'])
        self.labels = torch.load(data_config['labels'])

        if indices is not None:
            self.thermal = self.thermal[indices]
            self.pose = self.pose[indices]
            self.pose_mask = self.pose_mask[indices]
            self.labels = self.labels[indices]

        self.thermal_transform = thermal_transform
        self.pose_augment_fn = pose_augment_fn
        self.modality = modality

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        # tensors already → just cast
        label = self.labels[idx].long()

        if self.modality == 'thermal':
            image = self.thermal[idx].float()
            if self.thermal_transform:
                image = self.thermal_transform(image)
            return image, label

        elif self.modality == 'pose':

            keypoints = self.pose[idx].float()      # [N,17,3]
            mask = self.pose_mask[idx].float()

            if self.pose_augment_fn:
                keypoints = self.pose_augment_fn(keypoints, mask)

            return keypoints, mask, label


### Data Loader Preparation

Builds training, validation, and test datasets with preprocessing, stratified splitting, and class-balanced sampling.


In [ ]:

def prepare_unimodal_dataloaders(cfg, modality="thermal", pose_augment_fn=None):

    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ]) if modality == "thermal" else None

    val_test_transform = transforms.Compose([
        transforms.Normalize(mean=[0.5], std=[0.5])
    ]) if modality == "thermal" else None

    # ======================================================
    # Load full dataset (NO transform for splitting)
    # ======================================================
    full_dataset = UnimodalDataset(cfg['data']['train'], modality=modality)

    labels = full_dataset.labels.numpy()

    val_ratio = cfg.get("val_split", 0.15)
    test_ratio = cfg.get("test_split", 0.15)

    # ======================================================
    #  STRATIFIED TRAIN / TEMP
    # ======================================================
    sss1 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_ratio + test_ratio,
        random_state=cfg.get("seed", 42)
    )

    train_idx, temp_idx = next(sss1.split(np.zeros(len(labels)), labels))

    # ======================================================
    #  STRATIFIED VAL / TEST
    # ======================================================
    temp_labels = labels[temp_idx]

    sss2 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=test_ratio / (val_ratio + test_ratio),
        random_state=cfg.get("seed", 42)
    )

    val_rel, test_rel = next(sss2.split(np.zeros(len(temp_labels)), temp_labels))

    val_idx = temp_idx[val_rel]
    test_idx = temp_idx[test_rel]

    # ======================================================
    # Create datasets WITH transforms
    # ======================================================
    train_dataset = UnimodalDataset(
        cfg['data']['train'],
        thermal_transform=train_transform,
        pose_augment_fn=pose_augment_fn,
        modality=modality,
        indices=train_idx
    )

    val_dataset = UnimodalDataset(
        cfg['data']['train'],
        thermal_transform=val_test_transform,
        modality=modality,
        indices=val_idx
    )

    test_dataset = UnimodalDataset(
        cfg['data']['train'],
        thermal_transform=val_test_transform,
        modality=modality,
        indices=test_idx
    )
    def debug_split(name, ds):
        u, c = torch.unique(ds.labels, return_counts=True)
        print(name, dict(zip(u.tolist(), c.tolist())))

    debug_split("TRAIN", train_dataset)
    debug_split("VAL", val_dataset)
    debug_split("TEST", test_dataset)

    labels = train_dataset.labels.long()
    num_classes = cfg["model"]["classifier"]["num_classes"]

    class_sample_count = torch.tensor([
        (labels == i).sum().item()
        for i in range(num_classes)
    ])

    class_weights = 1. / class_sample_count.float()
    sample_weights = class_weights[labels]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    # ======================================================
    # Dataloaders
    # ======================================================
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg['training']['batch_size'],
        sampler=sampler
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg['training']['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg['training']['batch_size'],
        shuffle=False
    )

    return train_loader, val_loader, test_loader, class_weights.tolist(), full_dataset


#### Dataset Validation

In [ ]:


def check_label_alignment(train_loader, val_loader, test_loader):


    def collect(loader):
        labs = []
        for batch in loader:
            labs.extend(batch[-1].cpu().numpy())
        return set(labs)


    tr = collect(train_loader)
    va = collect(val_loader)
    te = collect(test_loader)


    if not (tr == va == te):
        print(" WARNING: Label sets differ across splits")
        print("Train:", tr)
        print("Val  :", va)
        print("Test :", te)



#### Data Augmentation and Preprocessing

In [ ]:


def pose_augment_fn(pose):
    return pose + torch.randn_like(pose) * 0.01



---

## Models

### Thermal Encoders

#### resnet series

In [ ]:

class ResNet18Encoder(nn.Module):
    def __init__(self, output_dim=64, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        resnet = resnet18(weights=weights)

        if pretrained:
            with torch.no_grad():
                new_weights = resnet.conv1.weight.mean(dim=1, keepdim=True)
                resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
                resnet.conv1.weight.copy_(new_weights)
        else:
            resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, output_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return self.fc(torch.flatten(x, 1))

class ResNet50Encoder(nn.Module):
    def __init__(self, output_dim=64, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        resnet = resnet50(weights=weights)

        if pretrained:
            with torch.no_grad():
                new_weights = resnet.conv1.weight.mean(dim=1, keepdim=True)
                resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
                resnet.conv1.weight.copy_(new_weights)
        else:
            resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.bn = nn.BatchNorm2d(2048)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(2048, output_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.bn(x)
        x = self.avgpool(x)
        return self.fc(torch.flatten(x, 1)
                       

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, 
groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.relu(x)

class DepthwiseCNN(nn.Module):
    def __init__(self, input_channels=1, output_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            DepthwiseSeparableConv(input_channels, 32),   # [B, 32, H, W]
            nn.MaxPool2d(2),                              # ↓

            DepthwiseSeparableConv(32, 64),               # [B, 64, H/2, W/2]
            nn.MaxPool2d(2),                              # ↓

            DepthwiseSeparableConv(64, 128),              # [B, 128, H/4, W/4]
            nn.MaxPool2d(2),                              # ↓

            DepthwiseSeparableConv(128, 256),             # [B, 256, H/8, W/8]
            nn.AdaptiveAvgPool2d((1, 1))                  # [B, 256, 1, 1]
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        x = self.encoder(x)
        return self.fc(x)


### Depth Estimation Encoder

class DepthEstimationEncoder(nn.Module):
    def __init__(self, output_dim=64, pretrained=True):
        super().__init__()
        self.backbone = deeplabv3_resnet50(pretrained=pretrained)

        # Adapt first conv to grayscale
        old_conv = self.backbone.backbone.conv1
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=False)
        with torch.no_grad():
            new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        self.backbone.backbone.conv1 = new_conv

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(2048, output_dim)

    def forward(self, x):
        features = self.backbone.backbone(x)
        if isinstance(features, dict):
            features = features["out"]
        pooled = self.pool(features)
        return self.fc(pooled.view(pooled.size(0), -1))





 #### Modern Thermal Encoders

In [ ]:



class MobileViTv2SmallEncoder(nn.Module):
    def __init__(self, output_dim=64, pretrained=True):
        super().__init__()
        self.model = timm.create_model("mobilevitv2_050", pretrained=pretrained, in_chans=1, num_classes=0)
        self.fc = nn.Linear(self.model.num_features, output_dim)

    def forward(self, x):
        x = self.model(x)
        return self.fc(x)

class EfficientFormerV2SEncoder(nn.Module):
    def __init__(self, output_dim=64, pretrained=True):
        super().__init__()
        self.model = timm.create_model("efficientformerv2_s0", pretrained=pretrained, in_chans=1, num_classes=0)
        self.fc = nn.Linear(self.model.num_features, output_dim)

    def forward(self, x):
        x = self.model(x)
        return self.fc(x)

class CoaTLiteEncoder(nn.Module):
    def __init__(self, output_dim=64, pretrained=True):
        super().__init__()
        self.model = timm.create_model("coat_lite_tiny", pretrained=pretrained, in_chans=1, num_classes=0)
        self.fc = nn.Linear(self.model.num_features, output_dim)

    def forward(self, x):
        x = self.model(x)
        return self.fc(x)

class ThermalSwinEncoder(nn.Module):
    def __init__(self, output_dim=64, pretrained=True):
        super().__init__()
        # Use Swin-Tiny with in_chans=1; can be swapped for IR-pretrained weights
        self.model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=pretrained, in_chans=1, num_classes=0)
        self.fc = nn.Linear(self.model.num_features, output_dim)

    def forward(self, x):
        x = self.model(x)
        return self.fc(x)


### Pose Encoders


#### CNN-based


In [ ]:

class PoseCNN1DEncoder(nn.Module):
    """
    Pose encoder using 1D CNNs over joints.
    Input: [B, N, J, 3] → Output: [B, D] using masked mean pooling.
    """
    def __init__(self, input_dim=3, embed_dim=64, num_joints=17):
        super().__init__()
        self.joint_embed = nn.Linear(input_dim, embed_dim)

        self.conv = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1)
        )

        self.embed_dim = embed_dim
        self.num_joints = num_joints

    def forward(self, x, pose_mask=None):
        """
        Args:
            x: Tensor of shape [B, N, J, 3]
            pose_mask: Optional tensor of shape [B, N] (1 = valid, 0 = padded)
        Returns:
            Tensor of shape [B, D]
        """
        B, N, J, _ = x.shape

        # Joint embedding
        x = self.joint_embed(x)                                # [B, N, J, D]
        x = x.view(B * N, J, self.embed_dim).transpose(1, 2)   # [B*N, D, J]
        x = self.conv(x).transpose(1, 2)                        # [B*N, J, D]
        x = x.view(B, N, J, self.embed_dim)                    # [B, N, J, D]

        # Pool over joints → [B, N, D]
        x = x.mean(dim=2)

        # Apply pose mask if provided
        if pose_mask is not None:
            pose_mask = pose_mask.unsqueeze(-1).float()        # [B, N, 1]
            x = x * pose_mask                                  # zero-out invalid entries
            summed = x.sum(dim=1)                              # [B, D]
            count = pose_mask.sum(dim=1).clamp(min=1e-6)       # [B, 1]
            x = summed / count                                 # masked average → [B, D]
        else:
            x = x.mean(dim=1)                                  # fallback average → [B, D]

        return x


#### GCN-based

In [ ]:
# ----------------------------
#  COCO JOINT ADJACENCY MATRIX
# ----------------------------

# COCO Keypoints Order (17):
# 0: Nose
# 1: Left Eye
# 2: Right Eye
# 3: Left Ear
# 4: Right Ear
# 5: Left Shoulder
# 6: Right Shoulder
# 7: Left Elbow
# 8: Right Elbow
# 9: Left Wrist
# 10: Right Wrist
# 11: Left Hip
# 12: Right Hip
# 13: Left Knee
# 14: Right Knee
# 15: Left Ankle
# 16: Right Ankle

def get_coco_adjacency(normalize=True):
    edges = [
        (0, 1), (0, 2), (1, 3), (2, 4),
        (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
        (5, 11), (6, 12), (11, 12),
        (11, 13), (13, 15), (12, 14), (14, 16)
    ]
    num_joints = 17
    adj = torch.zeros((num_joints, num_joints), dtype=torch.float32)
    for i, j in edges:
        adj[i, j] = 1
        adj[j, i] = 1  # Undirected graph

    if normalize:
        D = torch.diag(1.0 / (adj.sum(1) + 1e-6).sqrt())
        adj = D @ adj @ D  # Symmetric normalization

    return adj  # Shape: [17, 17]

# ----------------------------
# GCN Layer
# ----------------------------

class GraphConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(GraphConvLayer, self).__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        #x: [B, N, J, D], adj: [J, J]
        B, N, J, D = x.shape
        #print(f"Input x shape: {x.shape}")             # [B, N, J, D]
        #print(f"Adjacency matrix shape: {adj.shape}")  # [J, J]
        x = x.permute(0, 1, 3, 2)               # [B, N, D, J]
        #print(f"After permute to [B, N, D, J]: {x.shape}")
        x = torch.matmul(x, adj)                # [B, N, D, J]
        #print(f"After matmul with adj: {x.shape}")
        x = x.permute(0, 1, 3, 2)               # [B, N, J, D]
        #print(f"After permute back to [B, N, J, D]: {x.shape}")
        x = self.linear(x)                      # [B, N, J, out_dim]
        #print(f"After linear layer: {x.shape}")
        return torch.relu(x)



class PoseGCN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=64, num_layers=3, use_attention=True):
        super(PoseGCN, self).__init__()
        self.use_attention = use_attention
        self.num_layers = num_layers

        layers = []
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            out_dim_i = output_dim if i == num_layers - 1 else hidden_dim
            layers.append(GraphConvLayer(in_dim, out_dim_i))
        self.gcn_layers = nn.ModuleList(layers)

        if self.use_attention:
            self.attn = nn.Sequential(
                nn.Linear(output_dim * 17, 64),
                nn.Tanh(),
                nn.Linear(64, 1)
            )

    def forward(self, x, adj, pose_mask=None):
        """
        x: [B, N, 17, 3]       - (x, y, conf)
        adj: [17, 17]          - normalized adjacency matrix
        pose_mask: [B, N] or [B, N, 1]
        """
        B, N, J, D = x.shape
        x = x[..., :2]  # [B, N, 17, 2]

        for gcn in self.gcn_layers:
            x = gcn(x, adj)  # [B, N, 17, out_dim]

        x = x.reshape(B, N, -1)  # [B, N, 17*out_dim]

        if self.use_attention:
            attn_weights = self.attn(x).squeeze(-1)  # [B, N]
            if pose_mask is not None:
                pose_mask = pose_mask.float()
                attn_weights = attn_weights.masked_fill(pose_mask == 0, float('-inf'))
            attn_weights = torch.softmax(attn_weights, dim=-1).unsqueeze(-1)  # [B, N, 1]
            x = (attn_weights * x).sum(dim=1)  # [B, 17*out_dim]
            #print(f"After pose mask layer: {x.shape}")
        else:
            if pose_mask is not None:
                pose_mask = pose_mask.float().unsqueeze(-1)
                x = (x * pose_mask).sum(dim=1) / (pose_mask.sum(dim=1) + 1e-6)
                #print(f"After mean  layer: {x.shape}")
            else:
                x = x.mean(dim=1)
                #print(f"After mean  layer: {x.shape}")

        return x  # [B, 17*out_dim]


    
class PoseGCNWithConf(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=64, num_layers=3, use_attention=True):
        super().__init__()
        self.use_attention = use_attention
        self.num_layers = num_layers

        # Create GCN layers
        layers = []
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            out_dim = output_dim if i == num_layers-1 else hidden_dim
            layers.append(GraphConvLayer(in_dim, out_dim))
        self.gcn_layers = nn.ModuleList(layers)

        if self.use_attention:
            self.attn = nn.Sequential(
                nn.Linear(output_dim * 17, 64),
                nn.Tanh(),
                nn.Linear(64, 1)
            )

    def forward(self, x, adj, pose_mask=None):
        """
        x: [B, N, 17, 3] - (x, y, confidence)
        adj: [17, 17] - Adjacency matrix
        pose_mask: [B, N] - Mask for valid poses
        """
        B, N, J, D = x.shape
        
        # Separate coordinates and confidence
        coords = x[..., :2]  # [B, N, 17, 2]
        conf = x[..., 2:]    # [B, N, 17, 1]
        
        # Confidence weighting
        coords = coords * conf.expand_as(coords)
        
        # Process through GCN layers
        for gcn in self.gcn_layers:
            coords = gcn(coords, adj)  # [B, N, 17, out_dim]
        
        # Prepare for attention/pooling
        coords = coords.reshape(B, N, -1)  # [B, N, 17*out_dim]
        
        if self.use_attention:
            # Compute attention weights
            attn_weights = self.attn(coords).squeeze(-1)  # [B, N]
            
            # Apply mask if provided
            if pose_mask is not None:
                attn_weights = attn_weights.masked_fill(pose_mask == 0, float('-inf'))
            
            # Softmax and apply attention
            attn_weights = torch.softmax(attn_weights, dim=1).unsqueeze(-1)  # [B, N, 1]
            output = (attn_weights * coords).sum(dim=1)  # [B, 17*out_dim]
        else:
            # Simple pooling
            if pose_mask is not None:
                mask = pose_mask.float().view(B, N, 1)
                output = (coords * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-6)
            else:
                output = coords.mean(dim=1)
        
        return output  # [B, 17*output_dim]

#### MLP-based Pose Encoders


##### Joint-Level Pose Flatten


Projects flattened joint coordinates into a latent representation.
"

In [ ]:

class JointLevelPoseFlatten(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()
        self.projection = nn.Linear(51, output_dim)  # 17 joints * 3 coordinates

    def forward(self, x, pose_mask):
        # x shape: [B, N, 17, 3]
        B, N, num_joints, num_coords = x.shape
        
        # Flatten joint and coordinate dimensions only
        x = x.reshape(B, N, -1)  # [B, N, 51]
        
        # Apply mask (zero out padded frames)
        x = x * pose_mask.unsqueeze(-1)  # [B, N, 51]
        
        # Average across frames (ignoring padded frames)
        sum_features = x.sum(dim=1)  # [B, 51]
        valid_frames = pose_mask.sum(dim=1, keepdim=True)  # [B, 1]
        x = sum_features / (valid_frames + 1e-6)  # [B, 51]
        
        # Project to output dimension
        x = self.projection(x)  # [B, output_dim]
        return x


##### Pose MLP

MLP-based pose encoder with optional attention pooling.

In [ ]:

    
class PoseMLP(nn.Module):
    def __init__(self, input_dim=3, num_joints=17, hidden_dim=128, output_dim=64, num_layers=4, use_attention=True):
        super().__init__()
        self.use_attention = use_attention

        self.input_fc = nn.Linear(num_joints * input_dim, hidden_dim)

        self.hidden_layers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            for _ in range(num_layers - 2)
        ])

        self.output_fc = nn.Linear(hidden_dim, output_dim)

        if self.use_attention:
            self.attn = nn.Sequential(
                nn.Linear(output_dim, 64),
                nn.Tanh(),
                nn.Linear(64, 1)
            )

    def forward(self, x, pose_mask=None):
        # x: [B,N,17,3]
        B, N, J, D = x.shape

        x = x.reshape(B, N, J * D)
        x = self.input_fc(x)

        for layer in self.hidden_layers:
            x = layer(x)

        x = self.output_fc(x)  # [B,N,D]

        if self.use_attention:
            attn = self.attn(x).squeeze(-1)

            if pose_mask is not None:
                attn = attn.masked_fill(pose_mask == 0, float("-inf"))

            attn = torch.softmax(attn, dim=1).unsqueeze(-1)
            x = (attn * x).sum(dim=1)
        else:
            if pose_mask is not None:
                pose_mask = pose_mask.unsqueeze(-1)
                x = (x * pose_mask).sum(dim=1) / (pose_mask.sum(dim=1) + 1e-6)
            else:
                x = x.mean(dim=1)

        return x



#### Transformer-based Pose Encoders

##### SkeletonViT

Vision Transformer for skeleton representation learning.
"

In [ ]:


class SkeletonViT(nn.Module):
    def __init__(self, input_dim=3, embed_dim=64, num_joints=17, num_heads=4, num_layers=4, output_dim=64):
        """
        input_dim: 3 (x, y, conf)
        embed_dim: embedding dimension for each joint token
        num_joints: number of joints (e.g., 17 for COCO)
        num_heads: transformer attention heads
        num_layers: number of transformer encoder layers
        output_dim: output feature size for fusion/classifier
        """
        super(SkeletonViT, self).__init__()
        self.num_joints = num_joints

        # Joint embedding: each joint (x, y, conf) -> token
        self.joint_embed = nn.Linear(input_dim, embed_dim)

        # Learnable joint positional embeddings
        self.pos_embed = nn.Parameter(torch.randn(1, num_joints, embed_dim))

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output projection
        self.fc_out = nn.Linear(embed_dim * num_joints, output_dim)

    def forward(self, x, pose_mask=None):
        """
        x: [B, N, J, 3] - batch of skeletons (x, y, conf)
        pose_mask: [B, N] or [B, N, 1] optional, mask for valid persons
        """
        B, N, J, D = x.shape
        assert J == self.num_joints, f"Expected {self.num_joints} joints, got {J}"
        
        # Keep (x, y, conf)
        x = x.reshape(B * N, J, D)          # [B*N, J, 3]
        x = self.joint_embed(x) + self.pos_embed  # [B*N, J, embed_dim]

        # Transformer encoding
        x = self.transformer(x)             # [B*N, J, embed_dim]

        # Flatten joint tokens
        x = x.reshape(B, N, -1)             # [B, N, J*embed_dim]

        # Aggregate across N persons (with attention if mask provided)
        if pose_mask is not None:
            pose_mask = pose_mask.float()
            attn_weights = pose_mask / (pose_mask.sum(dim=1, keepdim=True) + 1e-6)  # [B, N]
            x = (x * attn_weights.unsqueeze(-1)).sum(dim=1)  # [B, J*embed_dim]
        else:
            x = x.mean(dim=1)

        # Final projection
        x = self.fc_out(x)                  # [B, output_dim]
        return x


##### PoseFormer

Transformer-based pose encoder for single-frame skeleton sequences.


In [ ]:


class PoseFormerSingleFrame(nn.Module):
    def __init__(self, input_dim=3, embed_dim=64, num_joints=17, num_heads=4, depth=4, mlp_ratio=4.0, output_dim=64):
        """
        input_dim: 3 (x, y, conf)
        embed_dim: token embedding dimension
        num_joints: number of joints (e.g., 17 for COCO)
        num_heads: multi-head attention heads
        depth: number of transformer encoder layers
        mlp_ratio: MLP expansion ratio
        output_dim: output feature size for fusion/classifier
        """
        super(PoseFormerSingleFrame, self).__init__()
        self.num_joints = num_joints

        # Joint embedding
        self.joint_embed = nn.Linear(input_dim, embed_dim)

        # Learnable joint position embeddings
        self.pos_embed = nn.Parameter(torch.zeros(1, num_joints, embed_dim))

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # Output projection
        self.fc_out = nn.Linear(embed_dim * num_joints, output_dim)

        # Initialization
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x, pose_mask=None):
        """
        x: [B, N, J, 3] - batch of skeletons (x, y, conf)
        pose_mask: [B, N] optional, mask for valid persons
        """
        B, N, J, D = x.shape
        assert J == self.num_joints, f"Expected {self.num_joints} joints, got {J}"

        # Flatten persons and embed joints
        x = x.reshape(B * N, J, D)               # [B*N, J, 3]
        x = self.joint_embed(x) + self.pos_embed # [B*N, J, embed_dim]

        # Apply transformer encoder
        x = self.transformer(x)                  # [B*N, J, embed_dim]

        # Flatten all joints
        x = x.reshape(B, N, -1)                   # [B, N, J*embed_dim]

        # Aggregate across N persons
        if pose_mask is not None:
            pose_mask = pose_mask.float()
            attn_weights = pose_mask / (pose_mask.sum(dim=1, keepdim=True) + 1e-6)
            x = (x * attn_weights.unsqueeze(-1)).sum(dim=1)  # [B, J*embed_dim]
        else:
            x = x.mean(dim=1)

        # Final projection
        x = self.fc_out(x)                        # [B, output_dim]
        return x


#### Hybrid

In [ ]:


# =====================
# Hybrid GCN + ViT
# ====================
class HybridGCNViT(nn.Module):
    """
    Hybrid Pose Encoder with GCN + Transformer (ViT-style) fusion.
    Input: [B, N, J, 3]  where last dim = (x, y, conf)
    Output: [B, D]
    """
    def __init__(self, input_dim=3, embed_dim=64, num_joints=17, num_heads=4):
        super().__init__()
        self.num_joints = num_joints
        self.embed_dim = embed_dim

        self.local_gcn = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim),
        )

        self.joint_pos_embed = nn.Parameter(torch.randn(1, num_joints, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x, person_mask=None):
        # x: [B, N, J, 3]
        B, N, J, C = x.shape
        assert C == 3, "Expected input with 3 channels per joint (x, y, conf)"
        x = x.view(B * N, J, C)

        # Extract joint-wise confidence for masking (conf in x[..., 2])
        joint_conf = x[:, :, 2]  # [B*N, J]
        joint_mask = (joint_conf > 0.05)  # valid if confidence > 0.05
        joint_mask = ~joint_mask  # Transformer expects 1 for padding

        # Apply GCN-like projection
        x = self.local_gcn(x)  # [B*N, J, D]
        x = x + self.joint_pos_embed  # Add joint positional encoding

        x = self.transformer(x, src_key_padding_mask=joint_mask)  # [B*N, J, D]
        x = x.mean(dim=1)  # [B*N, D]

        x = x.view(B, N, -1)  # [B, N, D]

        # Apply per-person mask (optional)
        if person_mask is not None:
            person_mask = person_mask.view(B, N, 1).float()  # [B, N, 1]
            x = x * person_mask
            x = x.sum(dim=1) / (person_mask.sum(dim=1) + 1e-6)  # [B, D]
        else:
            x = x.mean(dim=1)  # [B, D]

        return x


# =================
# PoseMixerEncoder
# =================

class PoseMixerEncoder(nn.Module):
    def __init__(self, input_dim=3, embed_dim=64, num_joints=17):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_joints = num_joints

        self.joint_embed = nn.Linear(input_dim, embed_dim)

        self.token_ln = nn.LayerNorm(embed_dim)
        self.token_fc = nn.Linear(num_joints, num_joints)

        self.channel_ln = nn.LayerNorm(embed_dim)
        self.channel_fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, pose_mask=None):
        """
        Args:
            x: [B, N, J, 3] = [batch, num_persons, num_joints, 3]
            pose_mask: [B, N] binary mask (1=valid, 0=invalid)
        Returns:
            out: [B, D] pose embedding per sample
        """
        x = self.joint_embed(x)              # [B, N, J, D]
        B, N, J, D = x.shape

        # Token mixing
        x = x.permute(0, 1, 3, 2)            # [B, N, D, J]
        x = x.transpose(-2, -1)              # [B, N, J, D]
        x = self.token_ln(x)                 # LayerNorm over D
        x = x.transpose(-2, -1)              # [B, N, D, J]
        x = self.token_fc(x)                 # Linear over J
        x = x.permute(0, 1, 3, 2)            # [B, N, J, D]

        # Channel mixing
        x = self.channel_ln(x)
        x = self.channel_fc(x)

        # Mean over joints
        x = x.mean(dim=2)                    # [B, N, D]

        if pose_mask is not None:
            pose_mask = pose_mask.unsqueeze(-1).float()  # [B, N, 1]
            x = x * pose_mask
            summed = x.sum(dim=1)                        # [B, D]
            count = pose_mask.sum(dim=1).clamp(min=1e-6) # [B, 1]
            out = summed / count
        else:
            out = x.mean(dim=1)                          # [B, D]

        return out



### Feature Projection

Projection layer used to project modality-specific features into a shared embedding space.

In [ ]:

class ProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)

### Classification Heads

In [ ]:

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128, dropout=0.3):
        super(MLPClassifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.fc(x)
    

class CosineClassifier(nn.Module):
    def __init__(self, input_dim=64, num_classes=10):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, input_dim))

    def forward(self, x):
        #print(x.shape)
        x = F.normalize(x, dim=1)
        w = F.normalize(self.weight, dim=1)
        return torch.matmul(x, w.t())  # [B, C]

class ResidualClassifier(nn.Module):
    def __init__(self, input_dim=64, num_classes=10):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, input_dim),
            nn.ReLU()
        )
        self.classifier = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        x = x + self.block(x)
        return self.classifier(x)

class TransformerClassifier(nn.Module):
    def __init__(self, input_dim=64, num_classes=10):
        super().__init__()
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=input_dim, nhead=4, batch_first=True),
            num_layers=2
        )
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # [B, 1, D]
        x = self.encoder(x)
        return self.fc(x.squeeze(1))  # [B, C]



### Loss Functions

In [ ]:
import torch
import torch.nn as nn

# ============================================================
# Safe Loss Switch Wrapper (smooth transition)
# ============================================================

class SafeSwitchLossWrapper(nn.Module):
    """
    Smoothly interpolates between two loss functions over N epochs.
    Used when switching loss phases.
    """

    def __init__(self, old_loss, new_loss, transition_epochs=10):
        super().__init__()
        self.old_loss = old_loss
        self.new_loss = new_loss
        self.transition_epochs = transition_epochs
        self.start_epoch = None

    def forward(self, logits, targets, epoch):

        # initialize local switch epoch
        if self.start_epoch is None:
            self.start_epoch = epoch

        # alpha ramps from 0 → 1 RELATIVE to switch
        alpha = min(1.0, (epoch - self.start_epoch) / self.transition_epochs)

        try:
            old_val = self.old_loss(logits, targets)
        except TypeError:
            old_val = self.old_loss(logits, targets, epoch)

        try:
            new_val = self.new_loss(logits, targets)
        except TypeError:
            new_val = self.new_loss(logits, targets, epoch)

        return (1 - alpha) * old_val + alpha * new_val


# ============================================================
# Safe Loss Revert Wrapper
# ============================================================

class SafeRevertLossWrapper(nn.Module):

    def __init__(self, old_loss, new_loss, transition_epochs=10):
        super().__init__()
        self.old_loss = old_loss
        self.new_loss = new_loss
        self.transition_epochs = transition_epochs
        self.start_epoch = None

    def forward(self, logits, targets, epoch):

        if self.start_epoch is None:
            self.start_epoch = epoch

        alpha = min(1.0, (epoch - self.start_epoch) / self.transition_epochs)

        try:
            old_val = self.old_loss(logits, targets)
        except TypeError:
            old_val = self.old_loss(logits, targets, epoch)

        try:
            new_val = self.new_loss(logits, targets)
        except TypeError:
            new_val = self.new_loss(logits, targets, epoch)

        return (1 - alpha) * old_val + alpha * new_val


In [ ]:
class LossSwitchManager:
    def __init__(self, trainer):
        self.t = trainer

    def unwrap(self, loss):
        while isinstance(loss, (SafeSwitchLossWrapper, SafeRevertLossWrapper)):
            loss = loss.new_loss
        return loss

    def samples_per_class(self):
        labels = torch.tensor(self.t.train_loader.dataset.labels)
        return [(labels == i).sum().item() for i in range(labels.max().item() + 1)]

    def minority_recall(self, report):
        recalls = []
        for v in report.values():
            if isinstance(v, dict) and "recall" in v:
                recalls.append(v["recall"])

        recalls.sort()
        return sum(recalls[:2]) / 2 if len(recalls) >= 2 else recalls[0]

    # ----------------------------------------
    # initial switch (on stagnation)
    # ----------------------------------------
    def maybe_initial_switch(self, epoch):

        if epoch >= 40 and self.t.loss_epochs_no_improve >= self.t.loss_switch_patience:

            metrics = self.t.evaluator.evaluate(self.t.val_loader, "Val", epoch)

            cm = metrics["confusion_matrix"]
            cost = generate_cost_matrix_from_confusion(cm)

            base = self.unwrap(self.t.criterion)

            self.t.criterion = SafeSwitchLossWrapper(
                base,
                SafeCostSensitiveCELoss(cost, alpha=0.0, transition_epochs=5),
                transition_epochs=5
            )

            self.t.loss_phase = "cost_sensitive"
            self.t.switch_epoch = epoch
            self.t.last_cost_update_epoch = epoch
            self.t.last_good_val_acc = metrics["accuracy"]
            self.t.loss_epochs_no_improve = 0

    # ----------------------------------------
    # refresh cost WITHOUT recreating module
    # ----------------------------------------
    def maybe_refresh_cost(self, epoch, metrics):

        if self.t.loss_phase == "cost_sensitive" and (epoch - self.t.last_cost_update_epoch) >= 10:

            cm = metrics["confusion_matrix"]
            cost = generate_cost_matrix_from_confusion(cm)

            if isinstance(self.t.criterion, SafeSwitchLossWrapper):
                if hasattr(self.t.criterion.new_loss, "cost_matrix"):
                    self.t.criterion.new_loss.cost_matrix = cost

            self.t.last_cost_update_epoch = epoch

    # ----------------------------------------
    # switch inside cost phase
    # ----------------------------------------
    def maybe_switch_inside_cost(self, metrics):

        if self.t.loss_phase != "cost_sensitive":
            return

        if self.t.loss_epochs_no_improve >= self.t.loss_switch_patience:

            minority = self.minority_recall(metrics["report"])
            base = self.unwrap(self.t.criterion)

            if minority < 0.6:

                self.t.criterion = SafeSwitchLossWrapper(
                    base,
                    LDAMLoss(cls_num_list=self.samples_per_class()),
                    10
                )

                self.t.loss_phase = "ldam"
                self.t.switch_epoch = self.t.current_epoch
                self.t.loss_epochs_no_improve = 0

            elif self.t.train_accuracies[-1] - self.t.val_accuracies[-1] > 0.10:

                self.t.criterion = SafeSwitchLossWrapper(
                    base,
                    LabelSmoothingCE(0.1),
                    10
                )

                self.t.loss_phase = "label_smooth"
                self.t.switch_epoch = self.t.current_epoch
                self.t.loss_epochs_no_improve = 0

    # ----------------------------------------
    # revert logic
    # ----------------------------------------
    def maybe_revert(self, epoch, val_acc):

        if self.t.switching_back and (epoch - self.t.revert_start_epoch) < 10:
            return

        self.t.switching_back = False

        if val_acc >= 0.85 * self.t.last_good_val_acc:
            return

        base = self.unwrap(self.t.criterion)

        if self.t.loss_phase == "cost_sensitive":
            new = LabelSmoothingCE(0.1)
            phase = "label_smooth"

        elif self.t.loss_phase == "label_smooth":
            new = LDAMLoss(self.samples_per_class())
            phase = "ldam"

        elif self.t.loss_phase == "ldam":
            new = ClassBalancedFocalLoss(self.samples_per_class())
            phase = "cb_focal"

        else:
            return

        self.t.criterion = SafeRevertLossWrapper(base, new, 10)

        self.t.loss_phase = phase
        self.t.revert_start_epoch = epoch
        self.t.switching_back = True
        self.t.last_good_val_acc = val_acc
        self.t.loss_epochs_no_improve = 0


In [ ]:
class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_class, beta=0.9999, gamma=2.0, smoothing=0.1):
        super().__init__()

        effective_num = 1.0 - torch.pow(beta, torch.tensor(samples_per_class, dtype=torch.float32))
        weights = (1.0 - beta) / effective_num
        alpha = weights / weights.sum()

        self.alpha = alpha.float()
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):

        # -------------------------------
        # Label-smoothed Cross Entropy
        # -------------------------------
        log_probs = F.log_softmax(inputs, dim=1)
        num_classes = inputs.size(1)

        one_hot = torch.zeros_like(inputs).scatter(1, targets.unsqueeze(1), 1)
        smoothed = one_hot * (1 - self.smoothing) + self.smoothing / num_classes

        ce_loss = -(smoothed * log_probs).sum(dim=1)

        # -------------------------------
        # Focal modulation
        # -------------------------------
        pt = torch.exp(-ce_loss)
        focal = (1 - pt) ** self.gamma * ce_loss

        # -------------------------------
        # Class-balanced weighting
        # -------------------------------
        alpha_t = self.alpha.to(inputs.device).gather(0, targets)
        focal = alpha_t * focal

        return focal.mean()


In [ ]:


class SafeCostSensitiveCELoss(nn.Module):
    def __init__(self, cost_matrix, alpha=0.0, transition_epochs=5):
        """
        cost_matrix: [C, C] tensor of misclassification costs
        alpha: initial weight for cost-sensitive term (0 = off, 1 = full)
        transition_epochs: number of epochs to fully switch from CE to cost-sensitive
        """
        super().__init__()
        self.register_buffer("cost_matrix", cost_matrix)
        self.alpha = alpha
        self.transition_epochs = transition_epochs
        self.start_epoch = None   # 🔑 local switch epoch

    def forward(self, logits, targets, epoch=None):

        # Standard CE
        ce_loss = F.cross_entropy(logits, targets, reduction='mean')

        # Cost-sensitive expected cost
        probs = F.softmax(logits, dim=1)                 # [B, C]
        costs_for_targets = self.cost_matrix[targets]   # [B, C]
        expected_cost = (probs * costs_for_targets).sum(dim=1).mean()
        if epoch is not None and self.transition_epochs > 0:

            if self.start_epoch is None:
                self.start_epoch = epoch

            blend = min(1.0, (epoch - self.start_epoch) / self.transition_epochs)

        else:
            blend = self.alpha

        return (1 - blend) * ce_loss + blend * expected_cost


In [ ]:


class LDAMLoss(nn.Module):
    def __init__(self, cls_num_list, max_m=0.5, weight=None, s=30):
        super().__init__()
        self.cls_num_list = cls_num_list
        m_list = 1.0 / torch.sqrt(torch.tensor(cls_num_list, dtype=torch.float32))
        m_list = m_list * (max_m / m_list.max())
        self.m_list = m_list
        self.s = s
        self.weight = weight

    def forward(self, x, target):
        if x.device != self.m_list.device:
            self.m_list = self.m_list.to(x.device)

        index = torch.zeros_like(x, dtype=torch.bool)  # changed to bool (was uint8, deprecated)
        index.scatter_(1, target.view(-1, 1), 1)

        batch_m = self.m_list[target].unsqueeze(1)
        x_m = x - self.s * batch_m * index.float()
        output = x * (~index) + x_m * index.float()

        return F.cross_entropy(self.s * output, target, weight=self.weight)


In [ ]:


# ========== Focal Loss ==========
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none')


    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss


        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            focal_loss = alpha_t * focal_loss


        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [ ]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        one_hot = torch.zeros_like(inputs).scatter(1, targets.unsqueeze(1), 1)
        smoothed = one_hot * (1 - self.smoothing) + self.smoothing / num_classes
        log_probs = torch.log_softmax(inputs, dim=1)
        return -(smoothed * log_probs).sum(dim=1).mean()


### ModelBuilder

In [ ]:
class ModelBuilder:

    def __init__(self, config, device):
        self.config = config
        self.device = device

    def build(self, modality, encoder_name, classifier_name):

        encoder_output_dim = self.config['model'].get('encoder_output_dim', 128)
        projection_output_dim = self.config['model'].get('projection_output_dim', 64)
        num_classes = self.config['model']['classifier']['num_classes']
        use_projection = self.config['model'].get('use_projection', False)
        # Force projection for GCN encoders
        if modality == "pose" and encoder_name.lower() in ["gcn", "gcn_conf"]:
            use_projection = True
        if modality == "pose" and encoder_name.lower() in ["gcn", "gcn_conf"]:
            assert use_projection, "GCN requires projection head"

        # Encoder
        if modality == "thermal":
            encoder = get_encoder(encoder_name, output_dim=encoder_output_dim)
            projection_input_dim = encoder_output_dim

        else:
            encoder = get_pose_module(encoder_name, output_dim=encoder_output_dim)

            if encoder_name.lower() in ["gcn", "gcn_conf"]:
                projection_input_dim = 17 * encoder_output_dim
            else:
                projection_input_dim = encoder_output_dim

        # Classifier
        classifier_input_dim = projection_output_dim if use_projection else projection_input_dim

        classifier = get_classifier(
            classifier_name,
            input_dim=classifier_input_dim,
            num_classes=num_classes
        )

        # Model
        if modality == "pose":
            adj = get_coco_adjacency(normalize=True).to(self.device)

            model = UnimodalModel(
                encoder=encoder,
                classifier=classifier,
                modality=modality,
                adj_matrix=adj,
                use_projection=use_projection,
                projection_input_dim=projection_input_dim,
                projection_output_dim=projection_output_dim,
                num_classes=num_classes
            ).to(self.device)

        else:
            model = UnimodalModel(
                encoder=encoder,
                classifier=classifier,
                modality=modality,
                use_projection=use_projection,
                projection_input_dim=projection_input_dim,
                projection_output_dim=projection_output_dim,
                num_classes=num_classes
            ).to(self.device)

        return model


### Model Factory

In [ ]:
def get_pose_module(name, input_dim=3, output_dim=64):
    name = name.lower()

    adj = get_coco_adjacency(normalize=True)

    pose_map = {
        "mlp": lambda: PoseMLP(output_dim=output_dim),

        "joint_level_2": lambda: JointLevelPoseFlatten(output_dim=output_dim),

        "pose_mixer": lambda: PoseMixerEncoder(embed_dim=output_dim),

        "pose_cnn1d": lambda: PoseCNN1DEncoder(embed_dim=output_dim),

        "poseformer": lambda: PoseFormerSingleFrame(output_dim=output_dim),

        "skeleton_vit": lambda: SkeletonViT(output_dim=output_dim),

        "hybrid_gcn_vit": lambda: HybridGCNViT(embed_dim=output_dim),

        "residual_gcn": lambda: ResidualPoseGCNEncoder(embed_dim=output_dim),

        "gcn": lambda: PoseEncoderWrapper(
            PoseGCN(output_dim=output_dim),
            adj
        ),

        "gcn_conf": lambda: PoseEncoderWrapper(
            PoseGCNWithConf(output_dim=output_dim),
            adj
        )
    }

    if name not in pose_map:
        raise ValueError(f"Unknown pose encoder: {name}")

    return pose_map[name]()


In [ ]:
def get_criterion(config, device, full_train_dataset):

    import numpy as np   # ← REQUIRED

    loss_type = config['training'].get('loss_type', 'ce')
    num_classes = config['model']['classifier']['num_classes']

    print("Sampler handles imbalance. Losses are frequency-neutral.")

    # real class counts (used by CB-Focal / LDAM)
    labels = torch.tensor(full_train_dataset.labels)
    samples_per_class = [(labels == i).sum().item() for i in range(num_classes)]

    # ==================================================
    #  LOSS SWITCH MODE → always start from CB-Focal
    # ==================================================
    if config["training"].get("enable_loss_switch", False):
        print(" Loss switch enabled → starting from CB-Focal")

        return ClassBalancedFocalLoss(
            samples_per_class=samples_per_class
        )

    # ==================================================
    # Normal loss selection
    # ==================================================
    if loss_type == 'focal':
        return FocalLoss(alpha=None, gamma=2.0)

    elif loss_type == 'cb_focal':
        return ClassBalancedFocalLoss(
            samples_per_class=samples_per_class
        )

    elif loss_type == 'label_smooth':
        return LabelSmoothingCE(smoothing=0.1)

    elif loss_type == 'ldam':
        return LDAMLoss(
            cls_num_list=samples_per_class,
            max_m=0.5,
            s=30,
            weight=None
        )

    elif loss_type == 'cost_sensitive':
        cost_tensor = generate_cost_matrix_from_confusion(
            np.eye(num_classes)
        )
        return SafeCostSensitiveCELoss(cost_tensor, alpha=0.0, transition_epochs=5)

    else:
        return nn.CrossEntropyLoss()


In [ ]:
class PoseEncoderWrapper(nn.Module):
    def __init__(self, encoder, adj=None):
        super().__init__()
        self.encoder = encoder
        self.adj = adj

    def forward(self, x, pose_mask=None):
        if self.adj is not None:
            return self.encoder(x, self.adj, pose_mask)
        else:
            return self.encoder(x, pose_mask)


In [ ]:



# ======================
# Model Builders
# ======================
def get_encoder(name, output_dim=128):
    encoder_map = {
        "resnet18": ResNet18Encoder,
        "resnet50": ResNet50Encoder,
        "vit": ViTEncoder,
        "swin_tiny": SwinTinyEncoder,
        "mobilevit": MobileViTEncoder,
        "depthwise_cnn": DepthwiseCNN,
        "depth_estimation": DepthEstimationEncoder,
        "convnext_tiny": ConvNeXtTinyEncoder,
        "convnextv2_tiny": ConvNeXtV2TinyEncoder,
        "efficientnetv2_b0": EfficientNetV2B0Encoder,
        "deit3_small": DeiTIIIEncoder,
        "mobilevitv2": MobileViTv2SmallEncoder,
        "efficientformerv2_s0": EfficientFormerV2SEncoder,
        "coat_lite": CoaTLiteEncoder,
        "levit256": LeViT256Encoder,
        "thermal_swin": ThermalSwinEncoder
    }




    name = name.lower()
    if name not in encoder_map:
        raise ValueError(f"Unknown encoder: {name}. Available: {list(encoder_map.keys())}")
    return encoder_map[name](output_dim=output_dim)



def get_classifier(name, input_dim, num_classes):
    if name.lower() == "mlp":
        return MLPClassifier(input_dim, num_classes)
    elif name.lower() == "transformer":
        return TransformerClassifier(input_dim, num_classes)
    elif name.lower() == "cosine":
        return CosineClassifier(input_dim, num_classes)
    elif name.lower() == "residual":
        return ResidualClassifier(input_dim, num_classes)
    elif name.lower() == "linear":
        return nn.Linear(input_dim, num_classes)
    else:
        raise ValueError(f"Unknown classifier: {name}")


---

## Training

#### Trainer

Manages model optimization, validation, checkpointing, and early stopping.

In [ ]:


class Trainer:

    def __init__(self, model, train_loader, val_loader, config, device,
                 criterion=None, checkpoint_path=None, start_epoch=0, test_loader=None):

        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.config = config
        self.epochs = config['training']['epochs']

        self.criterion = criterion 

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config['training']['lr'])
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode='max', patience=2, factor=0.5)

        self.train_metrics = []
        self.val_metrics = []

        self.best_accuracy = 0.0
        self.epochs_no_improve = 0
        self.loss_epochs_no_improve = 0
        self.patience = config['training'].get('patience', 10)
        self.loss_switch_patience = config['training'].get('loss_switch_patience', self.patience)
        self.min_delta = config['training'].get('min_delta', 0.001)
        self.min_epochs = config['training'].get('min_epochs', 0)

        self.loss_phase = "cb_focal"
        self.switch_epoch = None

        self.checkpoints = CheckpointManager(checkpoint_path, device)
        self.evaluator = Evaluator(self.model, device, self.criterion, config['training']['eval_path'])

        # ================= LOSS SWITCH PATCH =================
        self.enable_loss_switch = config["training"].get("enable_loss_switch", False)

        if self.enable_loss_switch:
            self.loss_manager = LossSwitchManager(self)

        self.current_epoch = 0
        self.train_accuracies = []
        self.val_accuracies = []

        self.last_cost_update_epoch = 0
        self.last_good_val_acc = 0.0
        self.switching_back = False
        self.revert_start_epoch = 0
        # =====================================================

        self.start_epoch = start_epoch

    def train(self):

        for epoch in range(self.start_epoch, self.epochs):

            self.current_epoch = epoch

            self.model.train()
            total_loss, correct, total = 0, 0, 0

            for batch in self.train_loader:

                if self.model.modality == "thermal":
                    images, labels = batch
                    images = images.to(self.device)
                    labels = labels.to(self.device)
                    outputs = self.model(images)

                elif self.model.modality == "pose":
                    keypoints, mask, labels = batch
                    keypoints = keypoints.to(self.device)
                    mask = mask.to(self.device)
                    labels = labels.to(self.device)
                    outputs = self.model(keypoints, mask)

                try:
                    loss = self.criterion(outputs, labels, epoch)
                except TypeError:
                    loss = self.criterion(outputs, labels)

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 5.0)
                self.optimizer.step()

                total_loss += loss.item() * labels.size(0)
                preds = torch.argmax(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

            train_acc = correct / total

            print(f"Epoch {epoch+1}/{self.epochs} - Loss: {total_loss/total:.4f}, Acc: {train_acc:.4f}")

            self.train_metrics.append({
                "epoch": epoch + 1,
                "loss": total_loss / total,
                "acc": train_acc
            })

            self.train_accuracies.append(train_acc)

            # =================================================
            # Validation
            # =================================================
            metrics = self.evaluator.evaluate(self.val_loader, "Val", epoch+1)

            # Dynamic cost refresh if wrapped
            if isinstance(self.criterion, SafeSwitchLossWrapper):
                cm = metrics["confusion_matrix"]
                new_cost = generate_cost_matrix_from_confusion(cm)
                if hasattr(self.criterion.new_loss, "cost_matrix"):
                    self.criterion.new_loss.cost_matrix = new_cost

            val_acc = metrics["accuracy"]

            self.val_metrics.append({
                "epoch": epoch + 1,
                "loss": metrics["val_loss"],
                "acc": val_acc
            })

            self.val_accuracies.append(val_acc)

            self.scheduler.step(val_acc)

            if val_acc - self.best_accuracy > self.min_delta:
                self.best_accuracy = val_acc
                self.epochs_no_improve = 0
                self.loss_epochs_no_improve = 0

                # SAVE BEST VAL ARTIFACTS
                self.evaluator.evaluate(self.val_loader, "Val", epoch+1, is_best=True)

                self.checkpoints.save(
                    epoch, self.model, self.optimizer, self.scheduler,
                    self.config, self.train_metrics, self.val_metrics,
                    self.best_accuracy, self.loss_phase, self.criterion, True
                )
            else:
                self.epochs_no_improve += 1
                self.loss_epochs_no_improve += 1
                if (
                    epoch + 1 >= self.min_epochs
                    and self.epochs_no_improve >= self.patience):
                    print(
                    f" Early stopping triggered "
                    f"(epoch={epoch+1}, "
                    f"min_epochs={self.min_epochs}, "
                    f"patience={self.patience})"
                    )
                    break

            # ================= LOSS SWITCH CORE =================
            if self.enable_loss_switch:
                self.loss_manager.maybe_initial_switch(epoch)
                self.loss_manager.maybe_refresh_cost(epoch, metrics)
                self.loss_manager.maybe_switch_inside_cost(metrics)
                self.loss_manager.maybe_revert(epoch, val_acc)

                print(f"[LOSS] phase={self.loss_phase} | criterion={type(self.criterion).__name__}")
            # ===================================================

            if (epoch + 1) % self.config['training'].get('checkpoint_freq', 10) == 0:
                self.checkpoints.save(
                    epoch, self.model, self.optimizer, self.scheduler,
                    self.config, self.train_metrics, self.val_metrics,
                    self.best_accuracy, self.loss_phase, self.criterion
                )

        self.evaluator.plot_metrics(self.train_metrics, self.val_metrics)

        # =================================================
        # Final Test Evaluation
        # =================================================
        if self.test_loader is not None:
            print(" Running final test evaluation...")
            test_metrics = self.evaluator.evaluate(self.test_loader, "Test", epoch=None)
            print("Final Test Accuracy:", test_metrics["accuracy"])

    def _get_samples_per_class(self):
        labels = torch.tensor(self.train_loader.dataset.labels)
        return [(labels == i).sum().item() for i in range(labels.max().item() + 1)]


#### Evaluator

Evaluates trained models and generates performance reports.

In [ ]:
class Evaluator:

    def __init__(self, model, device, criterion, eval_path):

        self.model = model
        self.device = device
        self.criterion = criterion
        self.eval_path = eval_path

    def evaluate(self, loader, name="Val", epoch=None, is_best=False):

        self.model.eval()

        all_preds, all_labels, all_features = [], [], []
        running_loss = 0.0

        with torch.no_grad():

            for batch in loader:

                if self.model.modality == "thermal":
                    inputs, labels = batch
                    inputs = inputs.to(self.device)
                    labels = labels.to(self.device)

                    outputs, feats = self.model(inputs, return_features=True)

                elif self.model.modality == "pose":
                    keypoints, mask, labels = batch
                    keypoints = keypoints.to(self.device)
                    mask = mask.to(self.device)
                    labels = labels.to(self.device)

                    outputs, feats = self.model(keypoints, mask, return_features=True)

                try:
                    loss = self.criterion(outputs, labels, epoch)
                except TypeError:
                    loss = self.criterion(outputs, labels)

                running_loss += loss.item() * labels.size(0)

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_features.append(feats.detach())

        avg_loss = running_loss / len(loader.dataset)

        metrics = compute_metrics(all_labels, all_preds)
        metrics["val_loss"] = avg_loss

        if is_best or name == "Test":

            self._save_confusion_matrix(metrics["confusion_matrix"], name)
            self._save_classification_report(metrics["report"], name)

            embeddings_path = os.path.join(self.eval_path, f"embeddings_{name.lower()}.pt")
            torch.save({
                "preds": np.array(all_preds),
                "labels": np.array(all_labels),
                "embeddings": torch.cat(all_features).cpu()
            }, embeddings_path)

        return metrics

    # =========================================================
    # Confusion Matrix
    # =========================================================

    def _save_confusion_matrix(self, cm, name):

        row_sum = cm.sum(axis=1, keepdims=True)
        row_sum[row_sum == 0] = 1
        cm_percentage = cm.astype(float) / row_sum * 100

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_percentage, annot=True, fmt=".2f", cmap='Blues')

        plt.title(f"{name} Confusion Matrix (%)")
        plt.savefig(
            os.path.join(self.eval_path, f"confusion_matrix_{name.lower()}.png")
        )
        plt.close()

    # =========================================================
    # Classification Report
    # =========================================================

    def _save_classification_report(self, report, name):

        df = pd.DataFrame(report).transpose()

        plt.figure(figsize=(10, 6))
        sns.heatmap(df.iloc[:-1, :-1], annot=True, fmt=".2f", cmap="YlGnBu")

        plt.title(f"{name} Classification Report")
        plt.savefig(
            os.path.join(self.eval_path, f"classification_report_{name.lower()}.png")
        )
        plt.close()

    # =========================================================
    # Metric Curves
    # =========================================================

    def plot_metrics(self, train_metrics, val_metrics):

        train_epochs = [m["epoch"] for m in train_metrics]
        train_acc = [m["acc"] for m in train_metrics]

        val_epochs = [m["epoch"] for m in val_metrics]
        val_acc = [m["acc"] for m in val_metrics]

        plt.figure(figsize=(10, 5))
        plt.plot(train_epochs, train_acc, label="Train Accuracy", marker='o')
        plt.plot(val_epochs, val_acc, label="Validation Accuracy", marker='o')
        plt.legend()
        plt.savefig(os.path.join(self.eval_path, "accuracy_curve.png"))
        plt.close()

        train_loss = [m["loss"] for m in train_metrics]
        val_loss = [m["loss"] for m in val_metrics]

        plt.figure(figsize=(10, 5))
        plt.plot(train_epochs, train_loss, label="Train Loss", marker='o')
        plt.plot(val_epochs, val_loss, label="Validation Loss", marker='o')
        plt.legend()
        plt.savefig(os.path.join(self.eval_path, "loss_curve.png"))
        plt.close()


### Checkpoint Management


#### Checkpoint Manager

Handles checkpoint saving and restoration.


In [ ]:


class CheckpointManager:
    def __init__(self, checkpoint_path, device):
        self.checkpoint_path = checkpoint_path
        self.device = device

    def save(self, epoch, model, optimizer, scheduler, config,
             train_metrics, val_metrics, best_accuracy,
             loss_phase, criterion, is_best=False):

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config,
            'train_metrics': train_metrics,
            'val_metrics': val_metrics,
            'best_accuracy': best_accuracy,
            'loss_phase': loss_phase,
            'criterion_state': self._extract_criterion_state(criterion)
        }

        # ===== always save last =====
        last_path = os.path.join(os.path.dirname(self.checkpoint_path), "checkpoint_last.pth")
        torch.save(checkpoint, last_path)
        print(f" Last checkpoint saved at epoch {epoch+1}")

        # ===== optional best =====
        if is_best:
            best_path = os.path.join(os.path.dirname(self.checkpoint_path), "checkpoint_best.pth")
            torch.save(checkpoint, best_path)
            print(f"New best model saved at epoch {epoch+1}")

    def load(self, model, optimizer, scheduler):

        checkpoint = torch.load(self.checkpoint_path, map_location=self.device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        return checkpoint

    def _extract_criterion_state(self, criterion):
        if isinstance(criterion, (SafeSwitchLossWrapper, SafeRevertLossWrapper)):
            return {
                'type': type(criterion).__name__,
                'old_loss': self._extract_criterion_state(criterion.old_loss),
                'new_loss': self._extract_criterion_state(criterion.new_loss)
            }
        elif hasattr(criterion, 'state_dict'):
            return criterion.state_dict()
        return None


### Performance Profiling

In [ ]:


def auto_profile_model(model, train_loader, modality, device, save_path="model_profile.txt"):
    """
    Profiles params, GFLOPs, model size, and avg inference time for a given model & modality.
    Uses REAL pose masks (no fake FLOPs).
    """

    model = model.to(device)
    model.eval()

    with torch.no_grad():

        batch = next(iter(train_loader))

        # ==========================
        # Prepare dummy inputs
        # ==========================
        if modality == "thermal":
            inputs, _ = batch
            inputs = inputs.to(device)
            dummy_input = inputs
            flops_input_shape = tuple(inputs.shape[1:])
            flops_model = model

        elif modality == "pose":
            keypoints, masks, _ = batch
            keypoints = keypoints.to(device)
            masks = masks.to(device)
            dummy_input = (keypoints, masks)

            # Wrapper that reuses REAL masks for FLOPs
            class PoseWrapper(nn.Module):
                def __init__(self, base_model, real_mask):
                    super().__init__()
                    self.base_model = base_model
                    self.real_mask = real_mask

                def forward(self, x):
                    return self.base_model(x, self.real_mask[:x.size(0)])

            flops_model = PoseWrapper(model, masks)
            flops_input_shape = tuple(keypoints.shape[1:])

        else:
            raise ValueError(f"Unknown modality: {modality}")

        # ==========================
        # 1. Parameter count
        # ==========================
        total_params = sum(p.numel() for p in model.parameters()) / 1e6

        # ==========================
        # 2. FLOPs
        # ==========================
        try:
            macs, _ = get_model_complexity_info(
                flops_model,
                input_res=flops_input_shape,
                as_strings=False,
                print_per_layer_stat=False,
                verbose=False
            )
            gflops = macs / 1e9
        except Exception as e:
            print(f" FLOPs estimation failed: {e}")
            gflops = None

        # ==========================
        # 3. Model size (safe temp)
        # ==========================
        with tempfile.NamedTemporaryFile(delete=False) as tmp:
            torch.save(model.state_dict(), tmp.name)
            size_mb = os.path.getsize(tmp.name) / (1024 ** 2)
            os.unlink(tmp.name)

        # ==========================
        # 4. Inference timing
        # ==========================
        if device.type == "cuda":
            torch.cuda.synchronize()

        # Warmup
        for _ in range(10):
            if modality == "pose":
                model(*dummy_input)
            else:
                model(dummy_input)

        if device.type == "cuda":
            torch.cuda.synchronize()

        runs = 100
        start = time.time()

        for _ in range(runs):
            if modality == "pose":
                model(*dummy_input)
            else:
                model(dummy_input)

        if device.type == "cuda":
            torch.cuda.synchronize()

        avg_ms = (time.time() - start) / runs * 1000

    # ==========================
    # 5. Save results
    # ==========================
    lines = [
        f"Params: {total_params:.3f} M",
        f"GFLOPs: {gflops:.3f}" if gflops is not None else "GFLOPs: N/A",
        f"Size: {size_mb:.3f} MB",
        f"Avg inference: {avg_ms:.3f} ms ({device.type.upper()})"
    ]

    profile_str = "\n".join(lines)
    print(profile_str)

    with open(save_path, "w") as f:
        f.write(profile_str)

    return {
        "params_M": total_params,
        "GFLOPs": gflops,
        "size_MB": size_mb,
        "time_ms": avg_ms
    }


---
## Experiment Management

### Device Initialization

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG_CLASS6 = get_config(6)
CONFIG_CLASS8 = get_config(8)

### Configuration Resolver

In [ ]:
def resolve_loss_tag(config, loss_name=None):
    tr = config["training"]
    exp = config["experiments"]

    #  Dynamic loss switching
    if tr.get("enable_loss_switch", False):
        return "loss_switch"

    #  Loss ablation
    if tr.get("enable_loss_ablation", False):
        return loss_name if loss_name is not None else exp["losses"][0]

    # Fixed loss
    return tr.get("loss_type", exp["losses"][0])


### Directory Manager

In [ ]:

def create_multimodal_directories(
    thermal,
    pose,
    fusion,
    classifier,
    loss,
    config
):
    """
    directory structure:

    <BASE_FROM_CONFIG>/
        seed_x/
            thermal__pose/
                fusion/
                    classifier/
                        loss/
                            MODELS/
                            EVALUATION_RESULTS/
    """

    exp = config["experiments"]
    tr = config["training"]

    # -------------------------------------------------
    # Resolve BASE DIR + SEED
    # -------------------------------------------------
    BASE_DIR = config["paths"]["base_dir"]
    seed = config["seed"]

    # -------------------------------------------------
    # Fallback to FIRST if ablation OFF
    # -------------------------------------------------
    if not tr.get("enable_fusion_ablation", False):
        fusion = exp["fusion_strategies"][0]

    if not tr.get("enable_classifier_ablation", False):
        classifier = exp["classifiers"][0]

    # -------------------------------------------------
    # Compile thermal + pose into ONE key
    # -------------------------------------------------
    tp_key = f"{thermal}__{pose}"

    #-------------------------------------------------
    # Multi-Seed Directory Structure
    #-------------------------------------------------
    base_dir = os.path.join(
        BASE_DIR,
        f"seed_{seed}",
        tp_key,
        fusion,
        classifier,
        loss
    )

    model_dir = os.path.join(base_dir, "MODELS")
    result_dir = os.path.join(base_dir, "EVALUATION_RESULTS")

    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(result_dir, exist_ok=True)

    return base_dir, model_dir, result_dir

### Experiment Logging

In [ ]:



class RunLogger:
    """
    One-row-per-run experiment logger (UNI-MODAL, EVALUATION ONLY).

    Profiling is logged separately via log_profile_results().

    Tracks:
      - Encoder
      - Classifier
      - LossType
      - LossPhase
      - LossClass
      - Final metrics
    """

    def __init__(self, save_path):
        self.save_path = save_path

        # Uni-modal headers (no Fusion)
        self.headers = [
            "RunID",
            "Seed",
            "Modality",
            "Encoder",
            "Classifier",
            "LossType",
            "LossPhase",
            "LossClass",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Status",
            "Timestamp"
        ]

        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        # write header once
        if not os.path.exists(save_path):
            with open(save_path, "w") as f:
                f.write(tabulate([], headers=self.headers, tablefmt="grid") + "\n")

    def log_run(
        self,
        run_id,
        seed,
        modality,
        encoder,
        classifier,
        loss_type,
        loss_phase,
        criterion,
        metrics,
        status="OK"
    ):

        loss_class = type(criterion).__name__ if criterion is not None else "NA"

        row = [
            run_id,
            seed,
            modality,
            encoder,
            classifier,
            loss_type,
            loss_phase,
            loss_class,
            f"{metrics['accuracy']:.4f}",
            f"{metrics['precision']:.4f}",
            f"{metrics['recall']:.4f}",
            f"{metrics['f1_score']:.4f}",
            status,
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ]

        table = tabulate([row], headers=self.headers, tablefmt="grid")

        with open(self.save_path, "a") as f:
            f.write(table + "\n\n")

        print(" Run logged.")


### Progress Tracking

In [ ]:
def mark_as_completed(run_id):
    """Append run_id to progress file and update in-memory set."""
    with open(progress_file, "a") as f:
        f.write(run_id + "\n")
    completed.add(run_id)





### Experiment Runner

In [ ]:
# =========================================================
# TRAINING PREPARER
# =========================================================

class TrainingPreparer:

    def __init__(self, config, device):
        self.config = config
        self.device = device

    def prepare(self, model, modality, model_combo) 
        # ==================================================
        combo = model_combo.replace(f"{modality}_", "")
        encoder, classifier = combo.rsplit("_", 1)

        base_dir, model_dir, result_dir = create_directories(
            modality,
            encoder,
            classifier,
            self.config["training"].get(
                "loss_tag",
                self.config["training"]["loss_type"]
            ),
            self.config
        )

        self.config['training']['eval_path'] = result_dir
        self.config['training']['model_dir'] = model_dir

        checkpoint_path = os.path.join(model_dir, "checkpoint_last.pth")
        start_epoch = 0

        # --------------------------------------------------
        # Dataloaders
        # --------------------------------------------------
        train_loader, val_loader, test_loader, _, full_dataset = prepare_unimodal_dataloaders(
            self.config,
            modality=modality
        )

        # --------------------------------------------------
        # Profiling (saved separately)
        # --------------------------------------------------
        profile_stats = auto_profile_model(
            model=model,
            train_loader=train_loader,
            modality=modality,
            device=self.device,
            save_path=os.path.join(base_dir, "profile.txt")
        )

        # --------------------------------------------------
        # Loss (unchanged)
        # --------------------------------------------------
        criterion = get_criterion(self.config, self.device, full_dataset)

        check_label_alignment(train_loader, val_loader, test_loader)

        # --------------------------------------------------
        # Trainer
        # --------------------------------------------------
        trainer = Trainer(
            model,
            train_loader,
            val_loader,
            self.config,
            self.device,
            criterion,
            checkpoint_path=checkpoint_path,
            start_epoch=start_epoch,
            test_loader=test_loader
        )
        if os.path.exists(checkpoint_path):

            checkpoint = torch.load(checkpoint_path, map_location=self.device, weights_only=False)

            try:
                trainer.model.load_state_dict(checkpoint['model_state_dict'])
                trainer.start_epoch = checkpoint['epoch'] + 1

                if 'optimizer_state_dict' in checkpoint:
                    trainer.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

                if 'scheduler_state_dict' in checkpoint:
                    trainer.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

                trainer.train_metrics = checkpoint.get('train_metrics', [])
                trainer.val_metrics   = checkpoint.get('val_metrics', [])
                trainer.best_accuracy = checkpoint.get('best_accuracy', 0.0)
                trainer.loss_phase    = checkpoint.get('loss_phase', "cb_focal")

            except RuntimeError as e:
                print("Incompatible checkpoint detected — starting fresh.")
                print(e)

        return trainer, model_dir, result_dir, test_loader, profile_stats


In [ ]:


class Final:

    def __init__(self, config, device, run_logger):

        self.builder = ModelBuilder(config, device)
        self.preparer = TrainingPreparer(config, device)

        self.config = config
        self.device = device
        self.run_logger = run_logger
    def run(self, modality, encoder_name, classifier_name):

        model_combo = f"{modality}_{encoder_name}_{classifier_name}"
        print(model_combo)

        loss_tag = self.config["training"].get("loss_tag", self.config["training"]["loss_type"])

        # --------------------------------------------------
        # Build model
        # --------------------------------------------------
        model = self.builder.build(modality, encoder_name, classifier_name)

        # --------------------------------------------------
        # Prepare training
        # --------------------------------------------------
        trainer, model_dir, result_dir, test_loader, profile_stats = self.preparer.prepare(
            model,
            modality,
            model_combo
        )

        # --------------------------------------------------
        # Train
        # --------------------------------------------------
        trainer.train()

        # --------------------------------------------------
        # Final evaluation
        # --------------------------------------------------
        test_metrics = trainer.evaluator.evaluate(test_loader, "Test")

        # --------------------------------------------------
        # Profile logging (separate table / file)
        # --------------------------------------------------
        log_profile_results([
            model_combo,
            f"{profile_stats['params_M']:.3f}",
            f"{profile_stats['GFLOPs']:.3f}" if profile_stats["GFLOPs"] else "N/A",
            f"{profile_stats['size_MB']:.3f}",
            f"{profile_stats['time_ms']:.3f}",
            
        ], self.config["profile_log_path"]) 

        # --------------------------------------------------
        # Evaluation logging (RunLogger only)
        # --------------------------------------------------
        self.run_logger.log_run(
            run_id=model_combo,
            seed=self.config["seed"],
            modality=modality,
            encoder=encoder_name,
            classifier=classifier_name,
            loss_type=self.config["training"].get("loss_tag", self.config["training"]["loss_type"]),
            loss_phase=trainer.loss_phase,
            criterion=trainer.criterion,
            metrics=test_metrics,
            status="DONE"
        )


---
## Visualization

### Metrics Visualization

### Confusion Matrix

### Classification Report

### Embedding Visualization

## Inference


### Model Loading


### Prediction Pipeline

## Main

In [ ]:

if __name__ == "__main__":

    exp = config["experiments"]
    tr = config["training"]

    default_loss = exp["losses"][0]
    default_classifier = exp["classifiers"][0]

    # =====================================================
    # MULTI-SEED CONTROL
    # =====================================================

    if tr.get("enable_multiseed_ablation", False):
        seeds = config["multiseed"]["seeds"]
    else:
        seeds = [config["seed"]]

    # =====================================================
    # SAVE ROOT DIRECTORY ONCE
    # =====================================================

    root_dir = config["paths"]["base_dir"]

    # =====================================================
    # SEED LOOP
    # =====================================================

    for seed in seeds:

        print(f"\n{'='*80}")
        print(f"🌱 RUNNING SEED {seed}")
        print(f"{'='*80}")

        # -------------------------------------------------
        # Seed
        # -------------------------------------------------

        config["seed"] = seed
        set_global_seed(seed)

        # -------------------------------------------------
        # Seed-specific output directory
        # -------------------------------------------------

        seed_dir = os.path.join(
            root_dir,
            f"SEED-{seed}"
        )

        config["paths"]["base_dir"] = seed_dir

        os.makedirs(seed_dir, exist_ok=True)

        # -------------------------------------------------
        # Seed-specific logs
        # -------------------------------------------------

        progress_file = os.path.join(
            seed_dir,
            "completed_runs.txt"
        )

        summary_log_path = os.path.join(
            seed_dir,
            "results_summary.txt"
        )
        run_logger = RunLogger(summary_log_path)
        
        profile_log_path = os.path.join(
            seed_dir,
            "profile_summary.txt"
        )
        config["profile_log_path"] = profile_log_path
        
        # -------------------------------------------------
        # Reload completed runs for THIS seed
        # -------------------------------------------------

        completed = load_completed(progress_file)

        # -------------------------------------------------
        # Fresh Final object for seed
        # -------------------------------------------------

        final = Final(config, device ,run_logger)

        # =================================================
        # EXPERIMENT LOOP
        # =================================================

        for modality in exp["modalities"]:

            encoders = (
                exp["pose_encoders"]
                if modality == "pose"
                else exp["thermal_encoders"]
            )

            for encoder in encoders:

                classifier_list = (
                    exp["classifiers"]
                    if tr.get("enable_classifier_ablation", False)
                    else [default_classifier]
                )

                for classifier in classifier_list:

                    loss_list = (
                        exp["losses"]
                        if tr.get("enable_loss_ablation", False)
                        else [default_loss]
                    )

                    for loss in loss_list:

                        # -------------------------------------
                        # Loss selection
                        # -------------------------------------

                        tr["loss_type"] = loss

                        if (
                            tr.get("enable_loss_switch", False)
                            and loss != "loss_switch"
                        ):
                            tr["loss_tag"] = "loss_switch"
                        else:
                            tr["loss_tag"] = loss

                        # -------------------------------------
                        # Unique run identifier
                        # -------------------------------------

                        run_id = (
                            f"seed_{seed}_"
                            f"{modality}_"
                            f"{encoder}_"
                            f"{classifier}_"
                            f"{tr['loss_tag']}"
                        )

                        # -------------------------------------
                        # Skip completed runs
                        # -------------------------------------

                        if run_id in completed:
                            print("⏭️ Skipping:", run_id)
                            continue

                        print("\n🚀 RUN:", run_id)

                        try:

                            final.run(
                                modality,
                                encoder,
                                classifier
                            )

                            with open(progress_file, "a") as f:
                                f.write(run_id + "\n")

                            completed.add(run_id)

                        except Exception as e:

                            print(
                                f"❌ FAILED: {run_id}\n"
                                f"Reason: {e}"
                            )

                            continue

---
# 